In [1]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# 1. Données et découpage
df = pd.read_csv("../data/raw/aug_train.csv")
X = df.drop(columns=["target", "enrollee_id"])
y = df["target"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2. Conversions
def convertir_experience(s):
    return pd.to_numeric(s.replace({"<1": "0", ">20": "21"}))

def convertir_last_new_job(s):
    return pd.to_numeric(s.replace({"never": "0", ">4": "5"}))

def convertir_numeriques(df_):
    df_ = df_.copy()
    df_["experience"] = convertir_experience(df_["experience"])
    df_["last_new_job"] = convertir_last_new_job(df_["last_new_job"])
    return df_

# 3. Pipeline de préparation
ohe = lambda: OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocess = ColumnTransformer([
    ("num_mediane", Pipeline([
        ("conversion", FunctionTransformer(convertir_numeriques, feature_names_out="one-to-one")),
        ("imputation", SimpleImputer(strategy="median"))]), ["experience", "last_new_job"]),
    ("cat_inconnu", Pipeline([
        ("imputation", SimpleImputer(strategy="constant", fill_value="Inconnu")),
        ("encodage", ohe())]), ["company_size", "company_type", "gender", "education_level"]),
    ("cat_na", Pipeline([
        ("imputation", SimpleImputer(strategy="constant", fill_value="Non applicable")),
        ("encodage", ohe())]), ["major_discipline"]),
    ("cat_freq", Pipeline([
        ("imputation", SimpleImputer(strategy="most_frequent")),
        ("encodage", ohe())]), ["enrolled_university"]),
    ("echelle", StandardScaler(), ["city_development_index", "training_hours"]),
    ("binaire", ohe(), ["relevent_experience"]),
    ("ville", ohe(), ["city"]),
])

# 4. Le modèle final (v1), entraîné sur le TRAIN seulement
modele_final = Pipeline([
    ("prep", preprocess),
    ("modele", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
modele_final.fit(X_train, y_train)

# 5. Validation croisée et tableau des coefficients
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1"]

noms = modele_final.named_steps["prep"].get_feature_names_out()
coefs = modele_final.named_steps["modele"].coef_[0]
tab = pd.DataFrame({"colonne": noms, "coef": coefs}).sort_values("coef")

print(X_train.shape, X_test.shape)
print(tab.iloc[0].to_dict(), tab.iloc[-1].to_dict())

(15326, 12) (3832, 12)
{'colonne': 'ville__city_city_7', 'coef': -1.66876383303097} {'colonne': 'ville__city_city_155', 'coef': 1.297959380036301}


In [2]:
preprocess_v2 = clone(preprocess).set_params(
    ville=OneHotEncoder(handle_unknown="infrequent_if_exist",
                        min_frequency=100, sparse_output=False))
v2 = Pipeline([("prep", preprocess_v2),
               ("modele", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))])

s = cross_validate(v2, X_train, y_train, cv=cv, scoring=scoring)
print({m: f"{s['test_'+m].mean():.3f} ± {s['test_'+m].std():.3f}" for m in scoring})

{'accuracy': '0.763 ± 0.006', 'precision': '0.517 ± 0.009', 'recall': '0.743 ± 0.011', 'f1': '0.610 ± 0.007'}


In [3]:
v2.fit(X_train, y_train)
noms2 = v2.named_steps["prep"].get_feature_names_out()
coefs2 = v2.named_steps["modele"].coef_[0]
tab2 = pd.DataFrame({"colonne": noms2, "coef": coefs2}).sort_values("coef")

print(len(noms2))
print(tab2.head(8))
print(tab2.tail(8))

69
                                        colonne      coef
27  cat_inconnu__education_level_Primary School -1.012763
50                         ville__city_city_138 -0.752553
38              echelle__city_development_index -0.630917
61                          ville__city_city_65 -0.593187
48                         ville__city_city_116 -0.576856
12     cat_inconnu__company_type_Funded Startup -0.467597
23     cat_inconnu__education_level_High School -0.444646
49                         ville__city_city_136 -0.417067
                                  colonne      coef
25   cat_inconnu__education_level_Masters  0.457029
26       cat_inconnu__education_level_Phd  0.483138
42                   ville__city_city_100  0.587122
44                   ville__city_city_103  0.615670
22  cat_inconnu__education_level_Graduate  0.616809
52                   ville__city_city_160  0.639434
10      cat_inconnu__company_size_Inconnu  0.929536
55                    ville__city_city_21  0.972075


In [4]:
print(tab2[tab2["colonne"].str.contains("gender")])

                        colonne      coef
20     cat_inconnu__gender_Male -0.169887
19  cat_inconnu__gender_Inconnu -0.142179
18   cat_inconnu__gender_Female -0.085376
21    cat_inconnu__gender_Other  0.093205
